In [0]:
import configparser
config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')

In [0]:
folder=dbutils.fs.ls(config['VOLUME']['volume_path'])

new_files=[]
for folder in folder:
    files=dbutils.fs.ls(folder.path)
    for file in files:
        new_files.append({
            'file_name':file.name,
            'file_path':file.path,
             'source_name':folder.name
            })
        
# print(new_files)

new_files_df=spark.createDataFrame(new_files)
new_files_df.display()


processed_files=spark.table( "healthcare_catalog.metadata.file_tracker")\
    .select('source_name','file_name')
processed_files.display()

new_files_df=new_files_df.join(processed_files,['source_name','file_name'],'left_anti')
new_files_df.display()






Insertion of new files

In [0]:
new_files = new_files_df.collect()
print(new_files)

In [0]:
from pyspark.sql.functions import lit,current_timestamp,col
for file_data in new_files:
    print(file_data['file_path'])
    if('claims' in file_data["source_name"]):
      table_name=config["TABLES"][file_data["source_name"].replace("_claims/", "")+'_table']
    else:
        table_name=config["TABLES"][file_data["source_name"].replace("/", "")+'_table']
    print(table_name)

    table_schema=spark.read.table(table_name).schema

    increment_df=(spark.read\
                   .format('csv').option('header',True)\
                   .schema(table_schema)\
                       .load(file_data['file_path']))
    increment_df = increment_df.select([col(c) for c in increment_df.columns])
    
    increment_df.withColumn('ingestion_ts',current_timestamp())\
               .write\
               .format('delta')\
               .mode('append')\
               .saveAsTable(table_name)
    file_data_df=spark.createDataFrame([
          {"source_name": file_data.source_name,
            "file_name": file_data.file_name,
            "file_path": file_data.file_path,
            "status": "processed"}
    ])
    file_data_df.withColumn('processed_ts',current_timestamp())\
               .write\
               .format('delta')\
               .mode('append')\
               .saveAsTable('healthcare_catalog.metadata.file_tracker')


In [0]:
%sql
-- select distinct(date_format(ingestion_ts, 'dd MM yyyy')) from healthcare_catalog.bronze.beneficiary
-- order by (date_format(ingestion_ts, 'dd MM yyyy')) desc;


describe history healthcare_catalog.bronze.outpatient_claims;